# Contact inhibition and the mode of tumour evolution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/evolution_modes.ipynb)

*A new model, the research model `evo_steric` of earlier biolgca versions,
after Noble R, Burri D, Le Sueur C, Lemant J, Viossat Y, Kather JN,
Beerenwinkel N (2022). Spatial structure governs the mode of tumour
evolution. Nat Ecol Evol 6: 207–217.
[doi:10.1038/s41559-021-01615-9](https://doi.org/10.1038/s41559-021-01615-9)*

**Question.** Sequencing shows that tumours evolve in different *modes*:
some by one selective sweep after another, some by branching into many
coexisting clones, some almost neutrally. Noble et al. argue that the
spatial structure of the tumour and the way its cells disperse decide the
mode. Here the tumour consists of glands of limited size, and cells may stop
moving where the tissue is full (*contact inhibition of locomotion*). Does
that change how the tumour evolves?

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

import lgca
from lgca import zoo
from lgca.zoo import evolution_modes as modes
from lgca.zoo._clones import diversity_bounds, dominant_clone, largest_clones

zoo.parameter_table("evolution_modes")

## The model

Identity-based cells without volume exclusion on a hexagonal lattice. Every
node is a gland of capacity $K$. In every time step:

1. **Birth and death:** every cell dies with probability $r_d$ and divides
   with its own probability $r_b$, as long as its node has fewer than $K$
   cells: divisions beyond $K$ cells fail. A full node thus turns over like
   a Moran process, and the fittest cells win the free places most often.
   $K$ limits division only: cells that move in can crowd a gland beyond
   it, until deaths bring it back below $K$.
2. **Driver mutations:** a daughter acquires a driver with probability
   $r_m$: it founds a new clone, and its birth rate is multiplied by
   $f = 1.1$.
3. **Movement:** every cell picks a channel. A velocity channel pointing to
   a node of relative density $\rho = n/K$ has the weight $e^{-\alpha\rho}$,
   the rest channel $e^{\gamma}$. With *contact inhibition* $\alpha > 0$
   cells avoid moving into crowded glands; with a *resting bias*
   $\gamma > 0$ they prefer to rest, whatever their neighbours.

The two act alike in a full tissue, where every neighbour has $\rho = 1$,
and differently at its edge, where a contact-inhibited cell still moves out
into free space. The fraction of cells that move, against the density of
the neighbours:

In [ ]:
rho = np.linspace(0, 1, 101)
fig, axis = plt.subplots(figsize=(6, 3.5))
for alpha in (0, 2.5, 5, 7.5, 10):
    axis.plot(rho, [modes.moving_fraction(alpha, density=r) for r in rho], label=f"α = {alpha}")
axis.plot(rho, [modes.moving_fraction(0, gamma=5, density=r) for r in rho], "k--", label="γ = 5")
axis.set(xlabel="relative density of the neighbours ρ", ylabel="fraction of cells that move", yscale="log")
axis.legend(fontsize="small", ncol=2)
plt.show()
plt.close(fig)

The tumour grows from one full gland at the centre of a lattice of
40 × 40 glands of $K = 64$ cells, for 1000 steps (60 × 60 glands of 512
cells, the gland size of Noble et al., and 2000 steps with
`build_spec(full=True)`). We run it with increasing contact inhibition and,
for comparison, with a resting bias of the same strength in a full tissue
($\gamma = 5$ and $\alpha = 5$ both let 3.9 % of the cells of a full tissue
move). About a minute:

In [ ]:
for operator in modes.build_spec().dynamics.operators:
    print(operator)

cases = {"α = 0": {}, "α = 2.5": {"alpha": 2.5}, "α = 5": {"alpha": 5.0}, "α = 7.5": {"alpha": 7.5},
         "α = 10": {"alpha": 10.0}, "γ = 5": {"gamma": 5.0}}
runs, records = {}, {}
for label, parameters in cases.items():
    runs[label], records[label] = modes.record(modes.build_spec(**parameters), every=20)

## Invasion and evolution

Noble et al. describe a tumour's evolution by two indices: the mean number
of driver mutations per cell $n$ (counting the one that founded the tumour,
so $n = 1$ before any driver) and the clonal diversity $D = 1/\sum_i p_i^2$,
where $p_i$ are the frequencies of the clones ($D = k$ for $k$ clones of
equal size). Over time, and in the $(n, D)$ plane of their Fig 3b:

In [ ]:
colours = dict(zip(cases, ["tab:blue", "tab:cyan", "tab:green", "tab:olive", "tab:red", "k"]))
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), constrained_layout=True)
for label, record in records.items():
    style = {"color": colours[label], "ls": "--" if label.startswith("γ") else "-", "label": label}
    axes[0, 0].plot(record["steps"], record["cells"], **style)
    axes[0, 1].plot(record["steps"], record["r_b"], **style)
    axes[1, 0].plot(record["steps"], record["n"], **style)
    axes[1, 1].plot(record["n"], record["D"], **style)
    axes[1, 1].plot(record["n"][-1], record["D"][-1], "o", color=colours[label])
axes[0, 0].set(xlabel="time step $k$", ylabel="cells", title="invasion")
axes[0, 1].set(xlabel="time step $k$", ylabel="mean birth rate $r_b$", title="selection")
axes[1, 0].set(xlabel="time step $k$", ylabel="drivers per cell $n$", title="drivers")
n = np.linspace(1, 3.5, 500)
bounds = diversity_bounds(n)
axes[1, 1].fill_between(n, np.minimum(bounds["upper"], 100), 100, color="0.9", label="not attainable")
axes[1, 1].plot(n, bounds["sweeps"], color="0.5", lw=1, label="one sweep after another")
axes[1, 1].set(xlabel="drivers per cell $n$", ylabel="clonal diversity $D$", yscale="log", ylim=(0.9, 60),
               xlim=(0.95, 3.5), title="modes of evolution (Noble et al., Fig 3b)")
axes[0, 0].legend(fontsize="small")
axes[1, 1].legend(fontsize="small", loc="upper left")
plt.show()
plt.close(fig)

for label, record in records.items():
    print(f"{label:7s}: {record['cells'][-1]:6d} cells, n = {record['n'][-1]:.2f}, D = {record['D'][-1]:5.1f}, "
          f"{record['clones'][-1]} clones, mean r_b = {record['r_b'][-1]:.3f}")

What the runs show:

- **Invasion.** With contact inhibition the tumour fills the lattice as
  fast as without it, or faster: cells at the edge still move out into
  free space. The resting bias slows invasion down.
- **Mixing drives sweeps.** Without contact inhibition ($\alpha = 0$) cells
  keep moving through the full tissue. A fitter clone spreads through the
  whole tumour, the next driver arises in it, and $n$ climbs while $D$ stays
  small: one sweep after another, the mode Noble et al. find in tumours
  without spatial structure.
- **Moderate contact inhibition makes the tumour branch.** With
  $\alpha = 2.5$ to 5 a new clone spreads only slowly beyond its gland.
  Many clones rise in different places and coexist, and $D$ grows large
  while $n$ still rises: *branching*, the mode Noble et al. associate with
  invasive glandular tumours. With $\alpha = 5$ the path runs along the
  edge of the attainable region: the clones are mostly one driver away
  from the initial clone, a star-shaped tree.
- **Strong contact inhibition halts evolution.** With $\alpha \ge 7.5$
  almost no cell moves in the full tissue. A driver can at best take over
  its own gland of 64 cells; it cannot spread further, and $n$ and $D$ stay
  close to 1, although as many drivers arise as in the other runs. Only the
  tumour's growth phase left room for clones to spread.
- **Resting is not the same as contact inhibition.** With $\gamma = 5$ as
  many cells move in a full tissue as with $\alpha = 5$, but the tumour
  fills the lattice later, and less happens by step 1000.

The dominant clone of every gland at the end (the nine clones that dominate
most glands in colour, all others in light grey):

In [ ]:
clone_colours = ListedColormap(["0.85"] + list(plt.get_cmap("tab10").colors[:9]))
shown = ("α = 0", "α = 5", "α = 10", "γ = 5")
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8), constrained_layout=True)
for axis, label in zip(axes, shown):
    lattice = runs[label].lgca
    clones = dominant_clone(lattice)
    lattice.plot_scalarfield(largest_clones(clones), ax=axis, tight_layout=False, cmap=clone_colours, vmin=-0.5,
                             vmax=9.5, cbar=False, mask=clones < 0)
    axis.set_title(label)
plt.show()
plt.close(fig)

And how the clones rose and fell, in Muller plots (each band a clone, drawn
inside the clone it descends from; clones that never held 1 % of the tumour
are left out):

In [ ]:
for label in ("α = 0", "α = 5", "α = 10"):
    fig, axis, *_ = runs[label].lgca.muller_plot(cutoff_rel=0.01)
    axis.set_title(label)
    fig.set_size_inches(10, 3)
    plt.show()
    plt.close(fig)

## Notes on the model

- **A new model.** Noble et al. simulate demes with a Moran process inside
  and dispersal between them by invasion or by gland fission. This model
  is an LGCA analogue: glands are nodes whose capacity stops division,
  dispersal is movement between nodes, and its dependence on crowding is
  contact inhibition. Unlike Noble's demes, a gland can hold more than $K$
  cells for a while, when cells move in: in a full tissue without contact
  inhibition about one gland in six is above $K$ (up to $1.3K$), with
  $\alpha = 5$ one in seventy, with $\alpha = 10$ none. It reproduces the paper's point, that the way cells disperse
  sets the mode, not its numbers: the tissues here hold 10⁵ cells, the
  paper's tumours 10⁶, and its driver effects are random.
- **The indices** are those of Noble et al., and the curves in the $(n, D)$
  plane are from their paper: tumours below the grey curve evolve by one
  complete sweep after another, and no tumour lies in the grey area,
  $D > 1/(2 - n)^2$ for $n < 2$. The paper classifies modes with a further
  curve (intermediate branching, in its supplement), and for tumours grown
  from a single cell; here we read the indices as a guide.
- **Earlier version.** The research model `evo_steric` of earlier biolgca
  versions used logistic division, $r_b(1 - n/K)$, with $r_d = 0.98\,r_b$,
  which keeps a gland at 2 % of $K$, and repulsion from the raw number of
  cells of a neighbour, which at $\alpha = 2$ already stops all movement in
  a full tissue of 64 cells per gland. Here division stops at the
  capacity, as in Noble et al., and contact inhibition acts on the
  relative density $n/K$, so that $\alpha$ means the same for every gland
  size.

## Explore it

The tumour with moderate contact inhibition, with the mean birth rate per
gland as the view. Move $\alpha$ and $\gamma$ while it grows and evolves:

In [ ]:
explorer = lgca.explore(
    modes.build_spec(alpha=5.0, size=30),
    {"dynamics.operators[1].terms[evolution_modes.contact_inhibition].beta": (0.0, 12.0),  # α
     "dynamics.operators[1].terms[resting_bias].beta": (0.0, 8.0)},  # γ
    view="mean r_b",
    measure=["population", "r_b"],
    steps_per_frame=5,
)
explorer

## Things to try

1. **Gland size.** Noble et al. find that larger glands give more sweeps
   within them. Run $\alpha = 5$ with $K = 16$ and $K = 256$ (and scale the
   lattice so that the tissue holds a similar number of cells).
2. **An established tissue.** `build_spec(start="full")` starts with every
   gland full of one clone: evolution without a growth phase. Which
   $\alpha$ still lets drivers sweep?
3. **Driver rate.** Raise $r_m$ tenfold. More drivers mean more clonal
   interference: does the tumour with $\alpha = 0$ still evolve by sweeps?
4. **Turnover.** Lower $r_d$. Without death, a full gland no longer turns
   over; how does that change the modes? (Noble et al. set the death rate
   in full demes much higher than the division rate.)